In [1]:
# importing libraries

import numpy as np
import pandas as pd
import duckdb
from collections import Counter, defaultdict
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [2]:
# pulling up updated session df

session_df = pd.read_pickle("data/intermediate_files/sessionized_events.pkl")

### Query Table creation

In [3]:
# query table for past data

def build_query_table_past(query_df, session_df, k=25, C=5):

    output_rows = []

    # group once for speed
    user_groups = {
        user_id: grp.sort_values("sequence_no").reset_index(drop=True)
        for user_id, grp in session_df.groupby("visitorid")
    }

    for query_id, (_, row) in enumerate(query_df.iterrows()):

        udf = user_groups[row["visitorid"]]

        hist_df = udf[udf.timestamp < row['timestamp']]

        last_k_items = set(hist_df['itemid'].drop_duplicates(keep='last').tail(k).tolist())
        last_C_categories = set(hist_df['category_id'].drop_duplicates(keep='last').tail(C).tolist())

        event_grouped = hist_df.groupby("event")

        past_events = event_grouped.size()

        # purchased items
        pre_purchased = set(event_grouped['itemid'].apply(list).get('transaction',[]))

        output_rows.append({
            "query_id": query_id,
            "visitorid": row["visitorid"],
            "session_id": row["session_id"],
            "sequence_no": row["sequence_no"],
            "event_pos_in_session": row["event_pos_in_session"],
            "query_time": row["timestamp"],
            "current_event": row["event"],
            "last_k_items": last_k_items,
            "last_C_categories": last_C_categories,
            "pre_purchased": pre_purchased,
            "past_view_count": past_events.get("view", 0),
            "past_atc_count": past_events.get("addtocart", 0),
            "past_order_count": past_events.get("transaction", 0)
        })

    return pd.DataFrame(output_rows)

In [4]:
def build_query_table_fut(query_df, session_fut_df, query_type):

    output_rows = []

    # group once for speed - past data
    user_groups = {
        user_id: grp.sort_values("sequence_no").reset_index(drop=True)
        for user_id, grp in session_fut_df.groupby("visitorid")
    }

    for query_id, (_, row) in enumerate(query_df.iterrows()):

        user_df = user_groups[row["visitorid"]]

        curr_sess = row["session_id"]
        curr_ts = row["timestamp"]

        # future positives within same session, using only all future interactions in the same session
        fut_df = user_df[
            (user_df["session_id"] == curr_sess) &
            (user_df["timestamp"] >= curr_ts)
            # & (user_df["event"] == "view")
        ]

        view_items = list(dict.fromkeys(fut_df.loc[fut_df.event == "view", "itemid"].tolist()))
        atc_items = list(dict.fromkeys(fut_df.loc[fut_df.event == "addtocart", "itemid"].tolist()))
        order_items = list(dict.fromkeys(fut_df.loc[fut_df.event == "transaction", "itemid"].tolist()))
        interaction_items = list(dict.fromkeys(fut_df["itemid"].tolist()))
        engagement_items = list(dict.fromkeys(atc_items + order_items))

        # while training, only keep those queries where we have orders in the future positives
        if query_type == 'train' and ((len(interaction_items) == 0) or (len(engagement_items) == 0)):
            continue

        if query_type != 'train' and (len(interaction_items) == 0):
            continue

        output_rows.append({
            "query_id": query_id,
            "future_positives": interaction_items,
            "future_view_items": view_items,
            "future_atc_items": atc_items,
            "future_order_items": order_items,
            "future_engagement_items": engagement_items
        })

    return pd.DataFrame(output_rows)

### CG feeders

#### User agnostic

In [5]:
## platform popularity based on orders basis 7 and 30 day popularity

def popularity_cg_base(size, session_past_df, d):

    max_time = session_past_df.timestamp.max()
    min_time = max_time - pd.DateOffset(days=d)

    orders_past_df = session_past_df.loc[((session_past_df.transactionid.notnull()) & (session_past_df.timestamp >= min_time))]

    item_pop = orders_past_df.groupby('itemid').size().sort_values(ascending=False)

    return item_pop.index.tolist()[:size], item_pop.reset_index()

In [6]:
## platform category popularity based on last 30 days

def category_cg_base(session_past_df, d, cat_rank, item_rank):

    max_time = session_past_df.timestamp.max()
    min_time = max_time - pd.DateOffset(days=d)

    orders_past_df = session_past_df.loc[((session_past_df.transactionid.notnull()) & (session_past_df.timestamp >= min_time))]
    
    cat_rank_query = f"""
    select 
        * 
        , row_number() over(partition by category_id order by total_orders desc) as item_rank
        , dense_rank() over(order by cat_orders desc) as cat_rank
    from (
    select 
        *, sum(total_orders) over(partition by category_id) as cat_orders from (
    select
        category_id, itemid, count(*) as total_orders
    from orders_past_df
    group by 1,2
    ) t1 ) t2
    order by cat_rank, item_rank
    """

    cat_rank_df = duckdb.sql(cat_rank_query).to_df()

    return cat_rank_df.loc[((cat_rank_df.cat_rank <= cat_rank) & (cat_rank_df.item_rank <= item_rank)),'itemid'].tolist(), cat_rank_df

In [7]:
## scores item basis their spread of users, interactions

def generality_CG(session_past_df, d, size):
    
    max_time = session_past_df.timestamp.max()
    min_time = max_time - pd.DateOffset(days=d)

    hist_df = session_past_df[(session_past_df.timestamp >= min_time)]

    generality_query = """
    select
        category_id
        , itemid

        , count(distinct visitorid) as unique_users
        , count(distinct cast(visitorid as varchar) || '-' || cast(session_id as varchar)) as unique_sessions
        , count(distinct date(timestamp)) as unique_days

        , (
            0.1*sum(case when event='view' then 1 else 0 end) 
            + 0.3*sum(case when event='addtocart' then 1 else 0 end) 
            + 0.6*sum(case when event='transaction' then 1 else 0 end)
        ) as weighted_interaction_score

    from hist_df
    group by 1, 2
    """

    item_stats = duckdb.sql(generality_query).to_df()

    for col in ['unique_users', 'unique_sessions', 'unique_days', 'weighted_interaction_score']:

        item_stats[f'{col}_pct'] = item_stats[col].rank(pct=True)

    item_stats['generality_score'] = (
        0.5*item_stats['weighted_interaction_score_pct']
        + 0.25*item_stats['unique_users_pct']
        + 0.15*item_stats['unique_sessions_pct']
        + 0.10*item_stats['unique_days_pct']
    )

    item_stats = duckdb.sql('select *, row_number() over(partition by category_id order by generality_score desc) as item_rank from item_stats').to_df()

    return item_stats.sort_values('generality_score', ascending=False)['itemid'].to_list()[:size], item_stats



#### User Based

In [8]:
## top items based on categories interacted by the user

def user_category_cg_base(input_q, cat_stats_df, item_rank):

    past_cats = input_q['last_C_categories']

    if not past_cats:
        return []

    return cat_stats_df.loc[(
        (cat_stats_df.category_id.isin(past_cats)) & (cat_stats_df.item_rank <= item_rank)
        ), 'itemid'].tolist()


In [9]:
def user_generality_cg_base(input_q, general_df, item_rank):

    past_cats = input_q['last_C_categories']

    if not past_cats:
        return []
    
    return general_df.loc[((general_df.category_id.isin(past_cats)) & (general_df.item_rank <= item_rank)), 'itemid'].tolist()

##### Co - event maps

In [10]:
## Co - interaction CG

# event_type : orders - returns visitorid and items that user bought over last 90 days - for co-order
# event_type : interactions - returns visitorid, session id, min timestamp and items in that session in list over last 30 days - for co-interacted

def build_event_item_table(events_df, event_type, d):

    max_time = events_df.timestamp.max()
    min_time = max_time - pd.DateOffset(days=d)

    if event_type=='orders':
        group = ["visitorid"]
        condition = (events_df.timestamp >= min_time)
        agg_condition = {
            "items": (
                "itemid",
                lambda x: list(dict.fromkeys(x.tolist()))
            )
        }
    
    else:
        group = ["visitorid", "session_id"]
        condition = ((events_df.timestamp >= min_time) & (events_df["event"].isin(["view", "addtocart","transaction"])))
        agg_condition = {
            "session_start": ("timestamp", "min"),

            "items": (
                "itemid",
                lambda x: list(dict.fromkeys(x.tolist()))
            )
        }

    browse_df = events_df.loc[condition].copy()

    event_items = (
        browse_df.groupby(group)
        .agg(**agg_condition)
        .reset_index()
    )

    return event_items

In [11]:
# item item co-event count stored 

def build_same_event_item_map(event_item_df, event_type, top_m, max_items_per_session=30):

    item_to_related = defaultdict(Counter)

    for items in event_item_df["items"]:
        if len(items) < 2:
            continue

        # optional cap to avoid huge sessions causing too much work
        if event_type == 'interactions' and len(items) > max_items_per_session:
            items = items[:max_items_per_session]

        for i, item in enumerate(items):
            other_items = items[:i] + items[i+1:]
            item_to_related[item].update(other_items)

    # keep only top M related items per item for fast query-time retrieval
    item_to_top_related = {
        item: counter.most_common(top_m)
        for item, counter in item_to_related.items()
    }

    return item_to_top_related

In [12]:
def co_event_cg(input_q, size, item_to_top_related):
    
    '''
    inputs - 
    input_q - pre-purchased list, last k interacted items (click/atc/order), past popular items of platform that the user hasn't bought yet
    size - size of output
    item_to_top_related - co-order map or co-interaction map

    output -
    recommender_list - array of recommended products
    '''
        
    already_purchased = input_q["pre_purchased"]
    seed_items = input_q["last_k_items"]

    if not seed_items:
        return []

    scores = Counter()

    for seed_item in seed_items:
        for related_item, cnt in item_to_top_related.get(seed_item, []):
            if related_item != seed_item and related_item not in already_purchased:
                scores[related_item] += cnt

    return [item for item, _ in scores.most_common(size)]

### Candidate Generation - v1

In [13]:
SOURCE_WEIGHTS = {
    "user_co_interaction": 1.00, "user_category": 0.75, "user_generality": 0.65,
    "global_category_pop_d7": 0.45, "global_category_pop_d30": 0.35,
    "global_pop_d7": 0.30, "global_pop_d30": 0.20, "global_generality_d30": 0.30,
}

def rank_decay_score(rank):
    return 1.0 / np.log2(rank + 1.0)

def dedupe_keep_order(items):
    if items is None: return []
    seen, out = set(), []
    for item in items:
        if pd.isna(item): continue
        item = int(item)
        if item not in seen:
            seen.add(item); out.append(item)
    return out

def add_source_scores(cg_scored, source_name, items, source_weight):
    for rank, item in enumerate(dedupe_keep_order(items), start=1):
        contrib = source_weight * rank_decay_score(rank)
        if item not in cg_scored:
            cg_scored[item] = {"raw_score": 0.0, "num_sources": 0, "sources": set()}
        cg_scored[item]["raw_score"] += contrib
        if source_name not in cg_scored[item]["sources"]:
            cg_scored[item]["num_sources"] += 1
            cg_scored[item]["sources"].add(source_name)

def build_combined_cg_list_for_row(row, pop_items_d7, pop_items_d30, cat_pop_items_d7,
                                   cat_pop_items_d30, gen_pop_items_d30, final_size=500):
    cg_scored = {}

    sources = [
        ("user_co_interaction", row["user_co_interaction_CG"]),
        ("user_category", row["user_cat_CG"]),
        ("user_generality", row["user_generality_CG"]),
        ("global_category_pop_d7", cat_pop_items_d7),
        ("global_category_pop_d30", cat_pop_items_d30),
        ("global_pop_d7", pop_items_d7),
        ("global_pop_d30", pop_items_d30),
        ("global_generality_d30", gen_pop_items_d30),
    ]

    for name, items in sources:
        add_source_scores(cg_scored, name, items, SOURCE_WEIGHTS[name])

    if not cg_scored: return [], [], []

    max_score = max(v["raw_score"] for v in cg_scored.values())
    scored = [
        {
            "itemid": item,
            "raw_score": info["raw_score"],
            "norm_score": info["raw_score"] / max_score if max_score > 0 else 0.0,
            "num_sources": info["num_sources"],
        }
        for item, info in cg_scored.items()
    ]

    scored = sorted(scored, key=lambda x: (-x["norm_score"], -x["raw_score"], -x["num_sources"], x["itemid"]))[:final_size]
    return [x["itemid"] for x in scored], [x["norm_score"] for x in scored], [x["raw_score"] for x in scored]

def combine_CG_for_metrics(query_input, session_train, final_size=500):
    query_input = query_input.copy()

    pop_items_d7, _ = popularity_cg_base(50, session_past_df=session_train, d=7)
    pop_items_d30, item_pop_df = popularity_cg_base(50, session_past_df=session_train, d=30)

    cat_pop_items_d7, cat_stats_d7_df = category_cg_base(session_past_df=session_train, d=7, cat_rank=5, item_rank=10)
    cat_pop_items_d30, _ = category_cg_base(session_past_df=session_train, d=30, cat_rank=5, item_rank=10)

    gen_pop_items_d30, item_stats_df = generality_CG(session_past_df=session_train, d=30, size=100)

    session_items_df = build_event_item_table(session_train, event_type="interactions", d=30)
    item_to_top_interacted_map = build_same_event_item_map(session_items_df, event_type="interactions", top_m=300, max_items_per_session=30)

    query_input["user_cat_CG"] = query_input.apply(user_category_cg_base, axis=1, cat_stats_df=cat_stats_d7_df, item_rank=20)
    query_input["user_generality_CG"] = query_input.apply(user_generality_cg_base, axis=1, general_df=item_stats_df, item_rank=20)
    query_input["user_co_interaction_CG"] = query_input.apply(co_event_cg, axis=1, size=200, item_to_top_related=item_to_top_interacted_map)

    cg_items, cg_norm_scores, cg_raw_scores = [], [], []
    for _, row in query_input.iterrows():
        items, norm_scores, raw_scores = build_combined_cg_list_for_row(
            row, pop_items_d7, pop_items_d30, cat_pop_items_d7, cat_pop_items_d30, gen_pop_items_d30, final_size
        )
        cg_items.append(items); cg_norm_scores.append(norm_scores); cg_raw_scores.append(raw_scores)

    query_input["combined_CG"] = cg_items
    query_input["combined_CG_norm_scores"] = cg_norm_scores
    query_input["combined_CG_raw_scores"] = cg_raw_scores
    
    return query_input, item_pop_df, item_stats_df

### Train and val/test data creation

In [14]:
artifact_ranges = {
    1: ("2015-05-03", "2015-06-01"),
    2: ("2015-05-15", "2015-06-15"),
    3: ("2015-06-01", "2015-07-01"),
    4: ("2015-06-15", "2015-07-15"),
    5: ("2015-07-01", "2015-08-01"),
    6: ("2015-07-15", "2015-08-15"),
    7: ("2015-08-01", "2015-09-01"),
}

query_ranges = {
    1: ("2015-06-01", "2015-06-15"),
    2: ("2015-06-15", "2015-07-01"),
    3: ("2015-07-01", "2015-07-15"),
    4: ("2015-07-15", "2015-08-01"),
    5: ("2015-08-01", "2015-08-15"),
    6: ("2015-08-15", "2015-09-01"),
    7: ("2015-09-01", "2015-09-15"),
}

engaged_sns = session_df.loc[
    session_df["event"].isin(["addtocart", "transaction"]),
    ["visitorid", "session_id"]
].drop_duplicates()

train_reduced_sess_df = session_df.merge(
    engaged_sns,
    on=["visitorid", "session_id"],
    how="inner"
)

arft_time, block_time = {}, {}
block_num = 7

for i in range(1, block_num+1):
    a_start, a_end = artifact_ranges[i]
    q_start, q_end = query_ranges[i]

    arft_time[i] = (
        (session_df["timestamp"] >= a_start) &
        (session_df["timestamp"] < a_end)
    )

    query_source_df = train_reduced_sess_df if i <= 5 else session_df

    block_time[i] = (
        (query_source_df["timestamp"] >= q_start) &
        (query_source_df["timestamp"] < q_end)
    )

### Query Inputs

In [15]:
query_inputs = {}
sample_rows = 30000
block_num = 7

for i in range(1, block_num+1):
    query_source_df = train_reduced_sess_df if i <= 5 else session_df
    df_query = query_source_df.loc[block_time[i]].copy()

    new_session = ((df_query["is_new_session"] == 1) & (df_query["session_id"] > 1))

    valid_q = df_query.loc[new_session].drop_duplicates(subset=["visitorid", "session_id", "timestamp"]).copy()

    query_base = valid_q.sample(n=min(sample_rows, valid_q.shape[0]),random_state=42 + i)

    # de_dup = query_base[["visitorid", "session_id"]].drop_duplicates()
    # temp_session = pd.merge(left=session_df, right=de_dup, on=['visitorid','session_id'], how='inner')
    print('query base shape: ', query_base.shape)

    query_ip_past = build_query_table_past(query_base, session_df)

    query_type = "train" if i <= 5 else "test"
    query_ip_fut = build_query_table_fut(query_base, df_query, query_type)

    query_input = pd.merge(left=query_ip_past, right=query_ip_fut, on="query_id", how="inner")

    query_inputs[i] = query_input.copy()

    print('query input shape: ', query_inputs[i].shape)

query base shape:  (1515, 12)
query input shape:  (1514, 18)
query base shape:  (1979, 12)
query input shape:  (1978, 18)
query base shape:  (1720, 12)
query input shape:  (1717, 18)
query base shape:  (2218, 12)
query input shape:  (2217, 18)
query base shape:  (1546, 12)
query input shape:  (1544, 18)
query base shape:  (30000, 12)
query input shape:  (30000, 18)
query base shape:  (30000, 12)
query input shape:  (30000, 18)


### Ranker Input Creation

In [16]:
def _as_list(x):
    """Convert list-like values to list; convert nulls to empty list."""
    if isinstance(x, (list, tuple, set, np.ndarray, pd.Series)):
        return list(x)
    if x is None:
        return []
    try:
        if pd.isna(x):
            return []
    except Exception:
        pass
    return [x]

def _dedupe(items):
    """Deduplicate item list while preserving order."""
    seen, out = set(), []
    for x in _as_list(items):
        try:
            if pd.isna(x):
                continue
        except Exception:
            pass
        x = int(x)
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def _fix_score_list(cands, scores):
    """Ensure score list length matches candidate list length."""
    cands = _as_list(cands)
    scores = _as_list(scores)
    return scores if len(scores) == len(cands) else [np.nan] * len(cands)

def _add_list_label(base, qdf, list_col, label_col):
    """Add binary label: 1 if candidate_itemid appears in qdf[list_col]."""

    # If source list column is absent, create all-zero label.
    if list_col not in qdf.columns:
        base[label_col] = 0
        return base

    # Keep query id and list column, then dedupe list values.
    pos = qdf[["global_query_id", list_col]].copy()
    pos[list_col] = pos[list_col].apply(_dedupe)

    # Convert list column to one item per row.
    pos = pos.explode(list_col).dropna()

    # If no positives exist, create all-zero label.
    if pos.empty:
        base[label_col] = 0
        return base

    # Rename exploded item column to candidate_itemid for joining.
    pos = pos.rename(columns={list_col: "candidate_itemid"})
    pos["candidate_itemid"] = pos["candidate_itemid"].astype("int64")

    # One positive marker per query-candidate pair.
    pos = pos.drop_duplicates(["global_query_id", "candidate_itemid"])
    pos[label_col] = 1

    # Join labels onto candidate rows.
    base = base.merge(
        pos[["global_query_id", "candidate_itemid", label_col]],
        on=["global_query_id", "candidate_itemid"],
        how="left"
    )

    # Missing means candidate was not in that future/source list.
    base[label_col] = base[label_col].fillna(0).astype("int8")
    return base

def _add_source_rank(base, qdf, source_col, source_name):
    """Add source flag and source rank for candidates from one CG source."""

    # If source list column is absent, create empty source features.
    if source_col not in qdf.columns:
        base[f"from_{source_name}"] = 0
        base[f"{source_name}_rank"] = np.nan
        return base

    # Keep query id and source candidate list.
    src = qdf[["global_query_id", source_col]].copy()
    src[source_col] = src[source_col].apply(_dedupe)

    # Convert source candidate list to one item per row.
    src = src.explode(source_col).dropna()

    # If source produced no candidates, create empty source features.
    if src.empty:
        base[f"from_{source_name}"] = 0
        base[f"{source_name}_rank"] = np.nan
        return base

    # Rename exploded item column to candidate_itemid.
    src = src.rename(columns={source_col: "candidate_itemid"})
    src["candidate_itemid"] = src["candidate_itemid"].astype("int64")

    # Rank candidates within each query for this source.
    src[f"{source_name}_rank"] = src.groupby("global_query_id").cumcount() + 1

    # Keep first rank if same item appears multiple times.
    src = src.drop_duplicates(["global_query_id", "candidate_itemid"], keep="first")

    # Join source rank onto final candidate rows.
    base = base.merge(
        src[["global_query_id", "candidate_itemid", f"{source_name}_rank"]],
        on=["global_query_id", "candidate_itemid"],
        how="left"
    )

    # Source flag = 1 if rank is present.
    base[f"from_{source_name}"] = base[f"{source_name}_rank"].notna().astype("int8")
    return base

def make_ranker_input_from_query_outputs(
    query_outputs,
    block_ids=None,
    split_name="train",
    candidate_col="combined_CG",
    norm_score_col="combined_CG_norm_scores",
    raw_score_col="combined_CG_raw_scores",
    max_candidates=None
):
    """
    Convert list-style query outputs into row-level ranker input.

    Input shape:
        one row per query with combined_CG = [item1, item2, ...]

    Output shape:
        one row per query_id × candidate_itemid
    """

    # Create default block ids if not passed.
    if block_ids is None:
        block_ids = list(range(1, len(query_outputs) + 1))

    all_rows, query_offset = [], 0

    # Process each rolling-time block separately.
    for df, block_id in zip(query_outputs, block_ids):
        qdf = df.copy().reset_index(drop=True)

        # Add block/split metadata.
        qdf["block_id"] = block_id
        qdf["split"] = split_name

        # Preserve local query_id if it exists, else use row index.
        qdf["local_query_id"] = qdf["query_id"] if "query_id" in qdf.columns else qdf.index

        # Create globally unique query id across blocks.
        qdf["global_query_id"] = np.arange(query_offset, query_offset + len(qdf))
        query_offset += len(qdf)

        # Ensure candidates and score columns are lists.
        qdf[candidate_col] = qdf[candidate_col].apply(_as_list)
        qdf[norm_score_col] = qdf.apply(
            lambda r: _fix_score_list(r[candidate_col], r.get(norm_score_col, [])),
            axis=1
        )
        qdf[raw_score_col] = qdf.apply(
            lambda r: _fix_score_list(r[candidate_col], r.get(raw_score_col, [])),
            axis=1
        )

        # Keep only useful query-level columns if present.
        keep_cols = [
            "global_query_id", "local_query_id", "block_id", "split",
            "visitorid", "session_id", "query_time", "timestamp", "current_event",
            "past_view_count", "past_atc_count", "past_order_count",
            "last_k_items", "last_C_categories", "pre_purchased",
            "future_positives", "future_view_items", "future_atc_items",
            "future_order_items", "future_engagement_items",
            "user_co_interaction_CG", "user_cat_CG", "user_generality_CG",
            candidate_col, norm_score_col, raw_score_col
        ]
        keep_cols = [c for c in keep_cols if c in qdf.columns]
        base = qdf[keep_cols].copy()

        # Explode candidate list and aligned score lists into row-level data.
        base = base.explode([candidate_col, norm_score_col, raw_score_col], ignore_index=True)

        # Rename candidate and score columns to ranker-friendly names.
        base = base.rename(columns={
            candidate_col: "candidate_itemid",
            norm_score_col: "cg_score_norm",
            raw_score_col: "cg_score_raw"
        })

        # Remove empty candidates and cast item ids.
        base = base.dropna(subset=["candidate_itemid"])
        base["candidate_itemid"] = base["candidate_itemid"].astype("int64")

        # Candidate rank = position inside combined_CG list.
        base["cg_rank"] = base.groupby("global_query_id").cumcount() + 1

        # Optionally keep only top-N candidates per query.
        if max_candidates is not None:
            base = base[base["cg_rank"] <= max_candidates].copy()

        # Add labels from future item lists.
        base = _add_list_label(base, qdf, "future_positives", "label_interaction")
        base = _add_list_label(base, qdf, "future_view_items", "label_view")
        base = _add_list_label(base, qdf, "future_atc_items", "label_atc")
        base = _add_list_label(base, qdf, "future_order_items", "label_order")
        base = _add_list_label(base, qdf, "future_engagement_items", "label_engagement")

        # Add useful history-match flags.
        base = _add_list_label(base, qdf, "pre_purchased", "candidate_previously_purchased")
        base = _add_list_label(base, qdf, "last_k_items", "candidate_in_last_k_items")

        # Add source flags and source ranks for user-specific CGs.
        base = _add_source_rank(base, qdf, "user_co_interaction_CG", "user_co_interaction")
        base = _add_source_rank(base, qdf, "user_cat_CG", "user_category")
        base = _add_source_rank(base, qdf, "user_generality_CG", "user_generality")

        # Remove list columns after converting them into labels/features.
        drop_list_cols = [
            "last_k_items", "last_C_categories", "pre_purchased",
            "future_positives", "future_view_items", "future_atc_items",
            "future_order_items", "future_engagement_items",
            "user_co_interaction_CG", "user_cat_CG", "user_generality_CG"
        ]
        base = base.drop(columns=[c for c in drop_list_cols if c in base.columns])

        all_rows.append(base)

    # Combine all blocks into one ranker dataframe.
    ranker_df = pd.concat(all_rows, ignore_index=True)

    # Create simple user-history bucket from past view count.
    if "past_view_count" in ranker_df.columns:
        ranker_df["user_history_bucket"] = pd.cut(
            ranker_df["past_view_count"],
            bins=[-1, 0, 5, 20, 100, np.inf],
            labels=["no_history", "very_low", "low", "medium", "warm"]
        ).astype(str)

    # Keep output sorted by query and CG rank.
    ranker_df = ranker_df.sort_values(["global_query_id", "cg_rank"]).reset_index(drop=True)
    return ranker_df

### Candidate Generation Output

In [17]:
block_num = 7
query_w_op = {}
item_pop_df = {}
item_gen_df = {}

for i in range(1,block_num+1):

    query_w_op[i], item_pop_df[i], item_gen_df[i] = combine_CG_for_metrics(query_inputs[i], session_df[arft_time[i]], final_size=500)

In [20]:
for i in range(1,block_num+1):
    print(query_w_op[i].shape)

(1514, 24)
(1978, 24)
(1717, 24)
(2217, 24)
(1544, 24)
(30000, 24)
(30000, 24)


In [34]:
def check_positives(row):

    future_eng_set = set(row['future_engagement_items'])
    cg_pred_set = set(row['combined_CG'])

    return len(cg_pred_set.intersection(future_eng_set))

In [56]:
ch_dfs = [query_inputs[k] for k in range(1,6)]
comb_ch_df = pd.concat(ch_dfs, axis=0, ignore_index=True) 

ch_df2 = pd.merge(left=session_df, right=comb_ch_df[['visitorid', 'session_id']], on=['visitorid', 'session_id'], how='inner')

ch_df2[ch_df2.event.isin(['addtocart'])]

# comb_ch_df.columns

,timestamp,visitorid,event,itemid,transactionid,sequence_no,is_new_session,session_id,start_st,end_st,event_pos_in_session,category_id
7,2015-06-01 00:19:30.010,123335,addtocart,109221,NaN,2,1,2.0,2015-06-01 00:19:30.010,2015-06-01 00:52:56.343,1,799
9,2015-06-01 00:20:32.209,1235469,addtocart,293126,NaN,8,0,2.0,2015-06-01 00:13:58.146,2015-06-01 00:26:29.117,5,324
12,2015-06-01 00:26:05.880,123335,addtocart,456005,NaN,3,0,2.0,2015-06-01 00:19:30.010,2015-06-01 00:52:56.343,2,1051
22,2015-06-01 00:42:14.347,572366,addtocart,130709,NaN,3,0,2.0,2015-06-01 00:41:34.616,2015-06-01 00:42:21.902,2,1051
24,2015-06-01 00:44:11.756,57905,addtocart,73593,NaN,590,0,33.0,2015-06-01 00:09:22.258,2015-06-01 02:09:22.692,8,1051
...,...,...,...,...,...,...,...,...,...,...,...,...
83893,2015-08-14 23:02:43.103,530559,addtocart,315543,NaN,3214,0,173.0,2015-08-14 22:15:42.154,2015-08-15 00:52:03.840,8,1279
83894,2015-08-14 23:02:48.948,1075788,addtocart,203231,NaN,11,0,5.0,2015-08-14 22:52:34.412,2015-08-14 23:06:04.557,5,84
83907,2015-08-14 23:23:05.874,530559,addtocart,104468,NaN,3220,0,173.0,2015-08-14 22:15:42.154,2015-08-15 00:52:03.840,14,491
83918,2015-08-14 23:41:01.583,410715,addtocart,248605,NaN,3,1,2.0,2015-08-14 23:41:01.583,2015-08-14 23:41:01.583,1,1098


In [57]:
comb_ch_df.future_order_items.apply(len).sum()
# .future_engagement_items.apply(len).sum()
# , 'transaction'

np.int64(7344)

In [37]:
# # we have around ~8970 postive queries with around ~23500 positives (orders + atc) - this shows 16190 have to check why this is less than ~24000
# # CG provides 5514 positives only being a major bottleneck to further training
# 23500 gets seperated in 16k atc + 7.5k orders 
# when we dedupe them, 14.5k atc + 7.3k orders are left
# but when we combine both and do the deduping, only 16.2k items remain
# out of which 5514 is being predicted by CG ~30%

# ch_dfs = [query_w_op[k] for k in range(1,6)] 
# comb_ch_df = pd.concat(ch_dfs, axis=0, ignore_index=True)

# comb_ch_df['positive_len'] = comb_ch_df.future_engagement_items.apply(len)

# # comb_ch_df['positive_len'].sum()
# comb_ch_df.shape

(8970, 25)

### Ranker Inputs

In [58]:
train_df = make_ranker_input_from_query_outputs([query_w_op[i] for i in range(1,6)], block_ids=[1,2,3,4,5], split_name="train", max_candidates=500)
val_df = make_ranker_input_from_query_outputs([query_w_op[6]], block_ids=[6], split_name="validation", max_candidates=500)
test_df = make_ranker_input_from_query_outputs([query_w_op[7]], block_ids=[7], split_name="test", max_candidates=500)

In [59]:
query_w_op[i]

,query_id,visitorid,session_id,sequence_no,event_pos_in_session,query_time,current_event,last_k_items,last_C_categories,pre_purchased,...,future_view_items,future_atc_items,future_order_items,future_engagement_items,user_cat_CG,user_generality_CG,user_co_interaction_CG,combined_CG,combined_CG_norm_scores,combined_CG_raw_scores
0,0,254307,2.0,4,1,2015-09-14 16:10:57.303,view,"{275889, 54826, 372196}",{228},{},...,[390287],[],[],[],"[432793, 79056, 340203, 222888, 17479]","[222888, 97307, 79056, 432793, 50411, 340203, ...","[461686, 345279, 119736, 99042, 465726, 279402...","[461686, 432793, 222888, 79056, 119736, 345279...","[1.0, 0.6841931671787891, 0.6463727797182931, ...","[1.5053347682418, 1.0299397627477056, 0.973007..."
1,1,434013,3.0,3,1,2015-09-12 21:04:56.474,view,"{389306, 78021}","{1483, 167}",{},...,[18178],[],[],[],"[320130, 438484, 416399, 108486, 169213, 57788...","[320130, 122604, 438484, 37029, 46130, 271896,...",[],"[320130, 438484, 416399, 122604, 338660, 10848...","[1.0, 0.6009456663291697, 0.42939447189974284,...","[1.902303842104968, 1.1431812499543097, 0.8168..."
2,2,681894,7.0,11,1,2015-09-12 17:30:02.222,view,"{113344, 344552, 83439, 165648, 153813, 159062...","{1233, 951, 1192, 685, 1563}",{},...,[110748],[],[],[],"[170262, 290999, 205702, 31791, 110077, 251594...","[344552, 224178, 111557, 131731, 438412, 38371...","[259202, 244717, 76060, 53139, 224178, 159581,...","[259202, 170262, 224178, 290999, 248455, 34455...","[1.0, 0.88, 0.7969571470559891, 0.659072737662...","[1.0, 0.88, 0.7969571470559891, 0.659072737662..."
3,3,784561,3.0,5,1,2015-09-09 22:10:49.221,view,"{447265, 21454, 47334}",{1429},{},...,[47334],[],[],[],[],"[47334, 450758, 257062, 338840, 361106, 230376...","[230376, 257062, 361106, 210018, 136119, 33884...","[230376, 257062, 361106, 47334, 450758, 338660...","[1.0, 0.7762101836027324, 0.6101771570213563, ...","[1.2315346716202145, 0.9559297535714575, 0.751..."
4,4,992458,2.0,3,1,2015-09-08 21:40:20.357,view,{190000},{5},{},...,[190000],[],[],[],"[190000, 255689, 190536, 301843, 368488, 289103]","[219512, 190000, 384302, 161623, 255689, 39302...","[219512, 255689, 368488, 11279, 359504, 131745...","[219512, 190000, 255689, 368488, 289103, 38430...","[1.0, 0.7803719389322562, 0.7670487737075262, ...","[1.840759009418101, 1.4364766772866226, 1.4119..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29995,1088492,4.0,5,1,2015-09-08 18:55:11.958,view,"{335328, 282419, 110403, 252596}","{1302, 361}",{},...,[181186],[],[],[],"[251396, 320900]","[1931, 35881, 252596, 460904, 154485, 282419, ...","[41169, 109663, 320900, 315309, 192221, 337196...","[41169, 320900, 251396, 109663, 1931, 338660, ...","[1.0, 0.9576849050639433, 0.8077659234441209, ...","[1.1707221977741757, 1.1211829768316126, 0.945..."
29996,29996,1056626,6.0,6,1,2015-09-01 17:57:57.211,view,{276868},{387},{},...,[276868],[],[],[],[],"[141172, 345179, 99020, 409300, 316114, 86464,...",[],"[141172, 338660, 461686, 320130, 248455, 65273...","[1.0, 0.9821413279813531, 0.7774381049873841, ...","[0.65, 0.6383918631878795, 0.5053347682417997,..."
29997,29997,1141585,5.0,6,1,2015-09-01 03:44:27.836,view,"{292720, 313856, 28392, 209118}","{342, 530}",{},...,[457377],[],[],[],"[444712, 341928, 218215, 328476, 194186, 22723...","[65273, 29100, 22436, 287664, 449571, 156173, ...","[28392, 313856]","[65273, 28392, 444712, 338660, 313856, 341928,...","[1.0, 0.8818717067826666, 0.7349977911472205, ...","[1.133951789482283, 1.0, 0.833452060536916, 0...."
29998,29998,1000753,2.0,2,1,2015-09-06 19:08:26.007,view,{457045},{1051},{},...,[48579],[],[],[],"[338660, 268755, 276762, 455183, 206317, 30552...","[455183, 234255, 454818, 231482, 13925, 118914...","[289633, 45140, 200214, 142608, 118382, 387028...","[338660, 455183, 289633, 276762, 268755, 45140...","[1.0, 0.8666641466692583, 0.6230302393337781, ...","[1.6050585298545463, 1.3910466811306046, 1.0, ..."


#### Getting Features for Deep NN ranker 

In [60]:
# add bucket

def add_item_bucket(df, item_df, col_name):

    distinct_blocks = df.block_id.unique().tolist()

    conc_dfs = []

    for b in distinct_blocks:
        item_df[b]['block_id'] = b
        conc_dfs.append(item_df[b])

    tp_df = pd.concat(conc_dfs, axis=0, ignore_index=True)
    upd_df = pd.merge(left=df, right=tp_df[['block_id','itemid','cum_bucket']], left_on=['block_id', 'candidate_itemid'], right_on=['block_id', 'itemid'], how='left')

    upd_df['cum_bucket'] = upd_df['cum_bucket'].fillna('90to100')
    upd_df = upd_df.rename(columns={'cum_bucket': f'{col_name}_bucket'})

    upd_df.drop(['itemid'], axis=1, inplace=True)

    return upd_df

##### popularity bucket

In [61]:
# update the column names for once and create the orders bucket
for i in range(1, block_num+1):
    
    item_pop_df[i].columns = ['itemid', 'orders']

    tp_df = item_pop_df[i]

    bucket_query = """
    select *,
    case when cum_oc <= 0.1 then '0to10'
        when cum_oc <= 0.2 then '10to20'
        when cum_oc <= 0.3 then '20to30'
        when cum_oc <= 0.4 then '30to40'
        when cum_oc <= 0.5 then '40to50'
        when cum_oc <= 0.6 then '50to60'
        when cum_oc <= 0.7 then '60to70'
        when cum_oc <= 0.8 then '70to80'
        when cum_oc <= 0.9 then '80to90'
    else '90to100' end as cum_bucket
    from (
    select
        *, sum(orders) over(order by orders desc rows between unbounded preceding and current row)*1.0000/sum(orders) over() as cum_oc 
    from
        tp_df
        ) as t1
    """

    item_pop_df[i] = duckdb.sql(bucket_query).to_df()


##### generality bucket

In [62]:
# add generality bucket

# update the column names for once and create the orders bucket
for i in range(1, block_num+1):

    tp_df = item_gen_df[i]

    bucket_query = """
    select *,
    case when cum_oc <= 0.1 then '0to10'
        when cum_oc <= 0.2 then '10to20'
        when cum_oc <= 0.3 then '20to30'
        when cum_oc <= 0.4 then '30to40'
        when cum_oc <= 0.5 then '40to50'
        when cum_oc <= 0.6 then '50to60'
        when cum_oc <= 0.7 then '60to70'
        when cum_oc <= 0.8 then '70to80'
        when cum_oc <= 0.9 then '80to90'
    else '90to100' end as cum_bucket
    from (
    select
        *, sum(generality_score) over(order by generality_score desc rows between unbounded preceding and current row)*1.0000/sum(generality_score) over() as cum_oc 
    from
        tp_df
        ) as t1
    """

    item_gen_df[i] = duckdb.sql(bucket_query).to_df()


##### Candidate category id and parent category id

In [63]:
cat_parent_map = pd.read_csv('data/intermediate_files/category_input.csv')

In [64]:
item_prop_df = pd.read_pickle('data/intermediate_files/item_properties.pkl')

In [65]:
# add columns
def safe_string_col(s, unknown_value="unknown"):
    return (
        s.astype("object")
         .where(s.notna(), unknown_value)
         .astype(str)
         .replace({"nan": unknown_value, "None": unknown_value, "<NA>": unknown_value})
    )

def add_cols(df, item_df):

    distinct_blocks = df.block_id.unique().tolist()

    conc_dfs = []

    for b in distinct_blocks:
        item_df[b]['block_id'] = b
        conc_dfs.append(item_df[b])

    tp_df = pd.concat(conc_dfs, axis=0, ignore_index=True)
    upd_df = pd.merge(left=df, right=tp_df[['block_id','itemid','categoryid', 'parent_category_id']], left_on=['block_id', 'candidate_itemid'], right_on=['block_id', 'itemid'], how='left')

    upd_df['categoryid'] = safe_string_col(upd_df['categoryid'])
    upd_df['parent_category_id'] = safe_string_col(upd_df['parent_category_id'])

    upd_df.drop(['itemid'], axis=1, inplace=True)

    return upd_df

In [66]:
item_cat_df = {}

for i in range(1, block_num+1):

    max_time = artifact_ranges[i][1]
    tp_df = item_prop_df[(item_prop_df.property == 'categoryid') & (item_prop_df.timestamp <= max_time)]

    cat_query = """
    select 
        itemid, a.value as categoryid, cast(b.parentid as int) as parent_category_id
    from (
    select
    *, row_number() over(partition by itemid order by timestamp desc) as rn
    from
    tp_df
    qualify rn = 1
    ) a
    left join cat_parent_map b on a.value = b.categoryid
    """

    item_cat_df[i] = duckdb.sql(cat_query).to_df()
    item_cat_df[i]['block_id'] = i


In [67]:
print(train_df.shape, val_df.shape, test_df.shape)

(3422822, 29) (8979560, 29) (9340931, 29)


In [68]:
# include the required columns for DNN features for train/test and val

train_df = add_item_bucket(train_df, item_gen_df, 'generality')
train_df = add_item_bucket(train_df, item_pop_df, 'popularity')
train_df = add_cols(train_df, item_cat_df)

val_df = add_item_bucket(val_df, item_gen_df, 'generality')
val_df = add_item_bucket(val_df, item_pop_df, 'popularity')
val_df = add_cols(val_df, item_cat_df)

test_df = add_item_bucket(test_df, item_gen_df, 'generality')
test_df = add_item_bucket(test_df, item_pop_df, 'popularity')
test_df = add_cols(test_df, item_cat_df)

In [69]:
print(train_df.shape, val_df.shape, test_df.shape)

(3422822, 33) (8979560, 33) (9340931, 33)


In [70]:
# train_df.label_engagement.sum()
query_inputs[5]

,query_id,visitorid,session_id,sequence_no,event_pos_in_session,query_time,current_event,last_k_items,last_C_categories,pre_purchased,past_view_count,past_atc_count,past_order_count,future_positives,future_view_items,future_atc_items,future_order_items,future_engagement_items
0,0,1044842,2.0,6,1,2015-08-05 23:01:33.513,transaction,"{16344, 294666, 349470}","{545, 1}",{},4,1,0,[349470],[],[],[349470],[349470]
1,1,1266239,3.0,4,1,2015-08-05 07:24:42.253,view,"{415928, 234035, 293005}","{1472, 1286, 936}",{},3,0,0,"[457705, 217552, 180269, 43268, 316135, 280985...","[457705, 217552, 180269, 280985, 115923, 37272...","[217552, 43268, 316135, 115923, 372728]",[],"[217552, 43268, 316135, 115923, 372728]"
2,2,861926,6.0,22,1,2015-08-13 03:09:01.583,view,"{277794, 45420, 121680, 44882, 350163, 75252, ...","{84, 1384, 618}",{},19,2,0,"[355985, 166333]","[355985, 166333]","[355985, 166333]",[],"[355985, 166333]"
3,3,853512,2.0,6,1,2015-08-10 23:50:38.816,view,"{204761, 171875, 229305}","{342, 1462, 1679}",{},5,0,0,[229305],[229305],[229305],[],[229305]
4,4,860026,12.0,14,1,2015-08-13 17:29:07.377,transaction,{146081},{928},{},12,1,0,[146081],[],[],[146081],[146081]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1539,1541,273910,3.0,24,1,2015-08-06 20:14:12.783,addtocart,"{324290, 17163, 460428, 279436, 306266, 140187...","{1593, 1441, 577, 624}","{279436, 324290, 17163, 140187}",15,4,4,"[48557, 165387, 322755]",[48557],"[48557, 165387, 322755]",[],"[48557, 165387, 322755]"
1540,1542,163156,2.0,18,1,2015-08-05 22:20:55.011,view,"{258944, 44001, 199810, 184320, 163039, 290946...","{1364, 624, 126, 577, 1240}",{146471},15,1,1,[146471],[146471],[146471],[146471],[146471]
1541,1543,1385073,29.0,489,1,2015-08-12 18:17:39.394,view,"{256769, 407687, 109583, 130709, 114840, 17568...","{1375, 746, 342, 959, 1173}","{369158, 29196, 109583, 257070, 78901, 439352,...",295,111,82,"[357583, 428065, 142008]","[357583, 142008]","[357583, 428065]",[],"[357583, 428065]"
1542,1544,31645,2.0,49,1,2015-08-04 03:52:39.774,transaction,"{445447, 79239, 326151, 292240, 27792, 314515,...","{1231, 1233, 387, 1694, 158}",{},38,10,0,"[179526, 436004, 270383]",[],[],"[179526, 436004, 270383]","[179526, 436004, 270383]"


In [71]:
meta_cols = [
    "categoryid",
    "parent_category_id",
    "popularity_bucket",
    "generality_bucket"
]

for name, df in [("TRAIN", train_df), ("VAL", val_df), ("TEST", test_df)]:
    print(f"\n{name}")
    print(df[meta_cols].isna().sum())
    print(df[meta_cols].nunique())
    print(df[meta_cols].head())

    for col in meta_cols:
        print(f"{col} top values:")
        print(df[col].astype(str).value_counts().head(10))


TRAIN
categoryid            0
parent_category_id    0
popularity_bucket     0
generality_bucket     0
dtype: int64
categoryid            901
parent_category_id    262
popularity_bucket      10
generality_bucket      10
dtype: int64
  categoryid parent_category_id popularity_bucket generality_bucket
0    unknown            unknown           90to100            10to20
1        342                500             0to10             0to10
2        342                500            10to20             0to10
3         48                500             0to10             0to10
4    unknown            unknown            20to30             0to10
categoryid top values:
categoryid
unknown    398624
1051       184956
959        165121
683        130760
1483       129444
342         93482
1613        83405
196         79828
5           67415
48          56562
Name: count, dtype: int64
parent_category_id top values:
parent_category_id
unknown    398624
955        226122
1606       203293
561        1825

### Eval Metrics

In [72]:
# Evaluation with HitRate, Recall, precision and MRR

def evalualtion_cg_query(true_op, pred_op, k):

    true_op_set, pred_op_set = set(true_op), set(pred_op[:k])

    common_items = true_op_set.intersection(pred_op_set)

    # hitrate
    hit_rate_k = 1 if len(common_items) > 0 else 0
    # recall
    recall_k = len(common_items)/len(true_op_set)
    # precision
    precision_k = len(common_items)/len(pred_op_set)
    #  MRR_k
    MRR_k = 0
    for rank, item in enumerate(pred_op[:k], start=1):
        if item in true_op_set:
            MRR_k = 1/rank
            break

    return {"hitrate_k":hit_rate_k, "recall_k":recall_k, "precision_k":precision_k, "MRR_k":MRR_k}

In [73]:
def evalualtion_cg(eval_df, k):

    metrics = defaultdict(list)
    len_df = eval_df.shape[0]

    for _, row in eval_df.iterrows():
        true_op = row['future_positives']
        pred_op = row['predicted_items']

        res = evalualtion_cg_query(true_op, pred_op, k)

        metrics['hitrate_k'].append(res['hitrate_k'])
        metrics['recall_k'].append(res['recall_k'])
        metrics['precision_k'].append(res['precision_k'])
        metrics['MRR_k'].append(res['MRR_k'])

    return {
        'mean_hitrate_k': sum(metrics['hitrate_k'])/len_df,
        'mean_recall_k': sum(metrics['recall_k'])/len_df,
        'mean_precision_k': sum(metrics['precision_k'])/len_df,
        'mean_MRR_k': sum(metrics['MRR_k'])/len_df
    }

In [74]:
block_num = 7
k = 500

for i in range(1,block_num+1):

    tp_df = pd.DataFrame(
    {'future_positives': query_w_op[i].future_positives, 
     'predicted_items': query_w_op[i].combined_CG
    })

    print(f'At i = {i}')
    print("Combined CG evalutaion : ",evalualtion_cg(tp_df, k=k))

    print('\n-------------------------------------------------------------------\n')

At i = 1
Combined CG evalutaion :  {'mean_hitrate_k': 0.6261558784676354, 'mean_recall_k': 0.43638039203819023, 'mean_precision_k': 0.003265707158954356, 'mean_MRR_k': 0.1271665017705175}

-------------------------------------------------------------------

At i = 2
Combined CG evalutaion :  {'mean_hitrate_k': 0.5910010111223458, 'mean_recall_k': 0.39352296596500635, 'mean_precision_k': 0.003081015061390854, 'mean_MRR_k': 0.12081962607574595}

-------------------------------------------------------------------

At i = 3
Combined CG evalutaion :  {'mean_hitrate_k': 0.6080372743156669, 'mean_recall_k': 0.413980887975907, 'mean_precision_k': 0.00300632358943405, 'mean_MRR_k': 0.1372087477354663}

-------------------------------------------------------------------

At i = 4
Combined CG evalutaion :  {'mean_hitrate_k': 0.5805142083897158, 'mean_recall_k': 0.3944780257931295, 'mean_precision_k': 0.002800638652890458, 'mean_MRR_k': 0.12207502530388051}

---------------------------------------

### Exporting ranker inputs

In [75]:
# storing in pkl files

train_df.to_pickle('data/intermediate_files/ranker_train_df.pkl')
val_df.to_pickle('data/intermediate_files/ranker_validation_df.pkl')
test_df.to_pickle('data/intermediate_files/ranker_test_df.pkl')

In [76]:
### Sanity Checks

def ranker_data_checks(df, name, label_col="label_engagement"):
    print(f"\n{name}")
    print("shape:", df.shape)
    print("queries:", df["global_query_id"].nunique())
    print("rows/query:", df.shape[0] / df["global_query_id"].nunique())
    print("max cg_rank:", df["cg_rank"].max())
    print("duplicate query-candidate rows:", df.duplicated(["global_query_id", "candidate_itemid"]).sum())

    label_cols = [
        c for c in [
            "label_interaction", "label_view", "label_atc",
            "label_order", "label_engagement"
        ]
        if c in df.columns
    ]

    print("\nlabel sums:")
    print(df[label_cols].sum())

    print("\nlabel rates:")
    print(df[label_cols].mean())

    q_pos = df.groupby("global_query_id")[label_col].sum()
    print(f"\nqueries with {label_col} positive:", (q_pos > 0).mean())

ranker_data_checks(train_df, "TRAIN")
ranker_data_checks(val_df, "VALIDATION")
ranker_data_checks(test_df, "TEST")


TRAIN
shape: (3422822, 33)
queries: 8970
rows/query: 381.58550724637684
max cg_rank: 500
duplicate query-candidate rows: 0

label sums:
label_interaction    10926
label_view            9373
label_atc             4570
label_order           2462
label_engagement      5514
dtype: int64

label rates:
label_interaction    0.003192
label_view           0.002738
label_atc            0.001335
label_order          0.000719
label_engagement     0.001611
dtype: float64

queries with label_engagement positive: 0.4931995540691193

VALIDATION
shape: (8979560, 33)
queries: 30000
rows/query: 299.3186666666667
max cg_rank: 500
duplicate query-candidate rows: 0

label sums:
label_interaction    12937
label_view           12689
label_atc              665
label_order            331
label_engagement       772
dtype: int64

label rates:
label_interaction    0.001441
label_view           0.001413
label_atc            0.000074
label_order          0.000037
label_engagement     0.000086
dtype: float64

querie